# Training a BERT classifier on CUAD

In [ ]:
import sys
sys.path.append('..')
from src.model import BertClassifier
from src.dataset import CuadDataset
from src.trainer import BertClassifierTrainer
import wandb
import json

## Data split

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import BertModel, BertTokenizer
import torch

df = pd.read_csv('../cuad_cls.csv')

train_df_temp, test_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df['Clause'])
train_df, val_df = train_test_split(train_df_temp, test_size=1/9, random_state=42, stratify=train_df_temp['Clause'])

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

train_dataset = CuadDataset(train_df, tokenizer)
val_dataset = CuadDataset(val_df, tokenizer)
test_dataset = CuadDataset(test_df, tokenizer)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=16, shuffle=False)

## Model setup

In [ ]:
model = BertModel.from_pretrained('bert-base-uncased')

configs = ['config1.json', 'config2.json', 'config3.json']

classes = [
    'Cap On Liability',
    'Other', 
    'Audit Rights',
    'Governing Law',
    'Exclusivity',
    'Ip Ownership Assignment', 
    'Non-Compete',
    'Termination For Convenience', 
    'Renewal Term',
    'Liquidated Damages'
]

trainer = BertClassifierTrainer(train_loader, val_loader, test_loader, classes)

results = []
    

## Experiments - 3 runs

In [ ]:
for config in configs:
    train_config = json.load(open(f'../configs/{config}'))
    wandb.init(
        project="clause-lens",
        config=train_config,
        id=train_config['exp_name'],
        reinit=True
    )

    results.append(trainer.train(BertClassifier(model, 10), train_config))


    wandb.finish()


## Model Selection

Let's compute which classifier performs better on the validation set:

In [ ]:
for i in range(0, 3):
    print(f"Classifier {i}: {results[i]['macro avg']}")

Classifier 2 is the best one

## Testing the best classifier

In [ ]:
config_2 = json.load(open('../configs/config2.json'))
wandb.init(
    project="clause-lens",
    config=config_2,
    id=config_2['exp_name'],
    resume="must"
)

In [ ]:
report = trainer.test(filename='../models/clause_bert_version_2_best.pt')

In [ ]:
report

In [ ]:
wandb.finish()